In [ ]:
# ===== Cell 0 =====
import os
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader, random_split
from torchaudio.transforms import Spectrogram  # 수정: Gammatonegram -> STFT Spectrogram
from tqdm import tqdm
from torchaudio.functional import add_noise
import glob

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SR = 8000 
N_FFT = 256             # 수정: 512 -> 256
N_BINS = N_FFT // 2 + 1 # 
HOP_LENGTH = 128        # 윈도우를 128샘플씩 이동
DURATION_SEC = 2        # 2초 단위로 잘라서 사용
BATCH_SIZE = 16 
EPOCHS = 20 
LR = 1e-3 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SpectrogramDataset(Dataset): 
    def __init__(self, clean_paths, noise_paths):
        self.clean_paths = clean_paths
        self.noise_paths = noise_paths
        self.max_len = SR * DURATION_SEC

        self.stft = Spectrogram(
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            power=1.0  # magnitude
        ).to(DEVICE)

    def __len__(self):
        return len(self.clean_paths)

    def load_wav(self, path):
        wave, sr = torchaudio.load(path)
        if wave.shape[0] > 1:  # 스테레오면 모노로
            wave = torch.mean(wave, dim=0, keepdim=True)
        if sr != SR:
            wave = torchaudio.functional.resample(wave, sr, SR)
        return wave

    def fix_len(self, wave):
        if wave.shape[1] > self.max_len:
            return wave[:, :self.max_len]
        return torch.nn.functional.pad(wave, (0, self.max_len - wave.shape[1]))

    def __getitem__(self, idx):
        clean = self.fix_len(self.load_wav(self.clean_paths[idx]))
        clean = clean + 1e-3 

        # 노이즈는 랜덤으로 하나 선택
        rand_idx = torch.randint(0, len(self.noise_paths), (1,)).item()
        noise = self.fix_len(self.load_wav(self.noise_paths[rand_idx]))
        noise = noise + 1e-3

        # -5 ~ 5dB 사이 랜덤 SNR로 mix
        snr = torch.FloatTensor(1).uniform_(-5, 5) 
        mixed = add_noise(clean, noise, snr)
        noise_scaled = mixed - clean 

        # STFT 스펙트로그램
        with torch.no_grad():
            c_gt = self.stft(clean.to(DEVICE))
            n_gt = self.stft(noise_scaled.to(DEVICE))
            m_gt = self.stft(mixed.to(DEVICE))

        # IRM 계산
        irm = torch.sqrt((c_gt ** 2) / (c_gt ** 2 + n_gt ** 2 + 1e-8))

        # (Time, Bins) 형태로 변환
        m_gt_2d = m_gt[0] #shape : (Freq_Bins, Time_Frames)
        irm_2d = irm[0]

        X = m_gt_2d.transpose(0, 1)
        Y = irm_2d.transpose(0, 1)

        return X.cpu(), Y.cpu()

In [ ]:
class SirenGRU(nn.Module):
    def __init__(self, input_size=N_BINS, hidden_size=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False  # 실시간 처리라 단방향
        )
        self.fc = nn.Linear(hidden_size, input_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.gru(x) # _ : 맨 마지막 순간의 내부 기억 상태 -> 필요 x
        return self.sigmoid(self.fc(out))


In [ ]:
if __name__ == "__main__":
    clean_files = glob.glob("/content/drive/MyDrive/dataset/clean_siren/*.wav")
    noise_files = glob.glob("/content/drive/MyDrive/dataset/wind/*.wav")

    if not clean_files or not noise_files:
        raise FileNotFoundError("wav 파일 x.")

    n_total = len(clean_files)
    n_val = int(n_total * 0.2)
    n_train = n_total - n_val 
    g = torch.Generator().manual_seed(42) 
    train_clean, val_clean = random_split(clean_files, [n_train, n_val], generator=g) 

    train_dataset = SpectrogramDataset(list(train_clean), noise_files)
    val_dataset = SpectrogramDataset(list(val_clean), noise_files)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = SirenGRU().to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for X, Y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
            X, Y = X.to(DEVICE), Y.to(DEVICE)

            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, Y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader) 

        model.eval()     
        val_total = 0.0                                
        with torch.no_grad():                          
            for X, Y in val_loader:                    
                X, Y = X.to(DEVICE), Y.to(DEVICE)      
                pred = model(X)                        
                val_total += criterion(pred, Y).item()
        val_loss = val_total / len(val_loader)         

        print(f"Epoch {epoch} | train: {train_loss:.5f} | val: {val_loss:.5f}") 

    torch.save(model.state_dict(), "siren_gru.pth")
    print("저장 완료")